In [0]:
-- Analysis: Regional YoY Revenue Growth
-- Purpose:
-- Measures year-over-year revenue growth by region
-- to identify growing and declining regional markets.

WITH regional_yearly_revenue AS (
    SELECT
        YEAR(f.order_date) AS sales_year,
        s.region,
        SUM(f.net_revenue) AS total_revenue

    FROM `end-to-end_pipeline`.gold.fact_sales f

    INNER JOIN `end-to-end_pipeline`.gold.dim_store s
        ON f.store_id = s.store_id

    GROUP BY
        YEAR(f.order_date),
        s.region
),

regional_growth AS (
    SELECT
        sales_year,
        region,
        total_revenue,

        LAG(total_revenue) OVER (
            PARTITION BY region
            ORDER BY sales_year
        ) AS previous_year_revenue

    FROM regional_yearly_revenue
)

SELECT
    sales_year,
    region,

    ROUND(total_revenue, 2) AS total_revenue,

    ROUND(previous_year_revenue, 2) AS previous_year_revenue,

    ROUND(
        ((total_revenue - previous_year_revenue)
        / NULLIF(previous_year_revenue, 0)) * 100,
        2
    ) AS revenue_growth_yoy_pct

FROM regional_growth

WHERE previous_year_revenue IS NOT NULL

ORDER BY
    sales_year,
    revenue_growth_yoy_pct DESC;